# Projet 02 — Chomage a Bruxelles : Genre, Niveau d'Etudes et Evolution Temporelle

**Question politique** : A Bruxelles-Capitale, les inegalites de genre dans le chomage varient-elles selon le niveau d'etudes, et quelles communes sont les plus vulnerables ?

**Hypothese** : Les femmes peu qualifiees sont davantage exposees au chomage que les hommes du meme profil. Cette vulnerabilite est amplifiee dans certaines communes specifiques.

---

**Auteur** : Kuate Joel Parfait  
**LinkedIn** : [joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
**Contact** : hello@dhcompany.pro | 0465 55 71 09  
**Source de donnees** : Open Data Bruxelles — opendata.brussels.be  
**Dataset** : `evolution-moyenne-annuelle-demandeurs-d-emploi-inoccupes-par-commune-genre-niveau-etudes-rbc`  
**Derniere mise a jour** : Mars 2026

---

> Kuate Joel Parfait — hello@dhcompany.pro | 0465 55 71 09  
> [linkedin.com/in/joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
> www.axiatechnologie.com

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
print("Librairies chargees.")

## 1. Chargement et exploration des donnees

In [ ]:
df = pd.read_csv('demandeurs_emploi_genre_etudes.csv', sep=';', encoding='utf-8-sig')
print("Shape:", df.shape)
print("Colonnes:", df.columns.tolist()[:10], "...")
df.head()

In [ ]:
# Nettoyage numerique
numeric_cols = [c for c in df.columns if c not in ['Année', 'Commune', 'Gemeente']]
for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '.').str.replace(' ', '').str.replace('%', ''), 
        errors='coerce')

print("Annees disponibles:", sorted(df['Année'].unique()))
print("Communes:", df['Commune'].nunique(), "communes")
df.describe()

## 2. Evolution temporelle du chomage par genre

In [ ]:
# Evolution annuelle totale pour Bruxelles-ville uniquement
df_bxl = df[df['Commune'].str.contains('Bruxelles|Brussel', case=False, na=False)]
df_region = df.groupby('Année').agg({
    'Total commune': 'sum',
    'Total hommes': 'sum',
    'Total femmes': 'sum'
}).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Graph 1 : Evolution par genre (region)
axes[0].fill_between(df_region['Année'], df_region['Total hommes'], 
                     alpha=0.6, color='steelblue', label='Hommes')
axes[0].fill_between(df_region['Année'], df_region['Total femmes'], 
                     alpha=0.6, color='coral', label='Femmes')
axes[0].set_title("Evolution du nombre de demandeurs d'emploi\npar genre — Bruxelles-Capitale", 
                  fontweight='bold')
axes[0].set_xlabel("Annee")
axes[0].set_ylabel("Nombre de demandeurs d'emploi")
axes[0].legend()
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Graph 2 : Part des femmes dans le total
df_region['pct_femmes'] = df_region['Total femmes'] / df_region['Total commune'] * 100
axes[1].plot(df_region['Année'], df_region['pct_femmes'], 
             color='coral', linewidth=2.5, marker='o', markersize=5)
axes[1].axhline(50, color='gray', linestyle='--', linewidth=1, label='Parite 50%')
axes[1].set_title("Part des femmes dans le total des\ndemandeurs d'emploi — Bruxelles-Capitale", 
                  fontweight='bold')
axes[1].set_xlabel("Annee")
axes[1].set_ylabel("% Femmes")
axes[1].set_ylim(40, 60)
axes[1].legend()
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('01_evolution_genre.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde.")

## 3. Analyse par niveau d'etudes

In [ ]:
# Evolution par niveau d'etudes (agregat regional)
df_etudes = df.groupby('Année').agg({
    "Total niveau d'etude ""faible""": 'sum',
    "Total niveau d'etude ""moyen""": 'sum',
    "Total niveau d'etude ""eleve""": 'sum'
}).reset_index() if any('faible' in c.lower() for c in df.columns) else None

# Identification des colonnes de niveaux d'etudes
col_faible = [c for c in df.columns if 'faible' in c.lower() and 'Total' in c and 'Hommes' not in c and 'Femmes' not in c]
col_moyen = [c for c in df.columns if 'moyen' in c.lower() and 'Total' in c and 'Hommes' not in c and 'Femmes' not in c]
col_eleve = [c for c in df.columns if ('lev' in c.lower() or 'elev' in c.lower()) and 'Total' in c and 'Hommes' not in c and 'Femmes' not in c]

print("Col faible:", col_faible)
print("Col moyen:", col_moyen)
print("Col eleve:", col_eleve)

In [ ]:
# Trouver les bonnes colonnes pour les niveaux d'etudes
niveaux = {}
for c in df.columns:
    cl = c.lower()
    if 'total' in cl and 'hommes' not in cl and 'femmes' not in cl and '%' not in c:
        if 'faible' in cl:
            niveaux['Niveau faible'] = c
        elif 'moyen' in cl:
            niveaux['Niveau moyen'] = c  
        elif chr(233)+'lev' in cl or 'lev' in cl:
            niveaux['Niveau eleve'] = c

print("Colonnes niveaux identifies:", niveaux)

if len(niveaux) >= 2:
    df_niv = df.groupby('Année')[list(niveaux.values())].sum().reset_index()
    df_niv.columns = ['Annee'] + list(niveaux.keys())
    
    fig, ax = plt.subplots(figsize=(13, 6))
    colors = ['#d62728', '#ff7f0e', '#2ca02c']
    for i, (label, col) in enumerate(zip(niveaux.keys(), df_niv.columns[1:])):
        ax.plot(df_niv['Annee'], df_niv[col], linewidth=2.5, marker='o', 
                markersize=5, color=colors[i], label=label)
    
    ax.set_title("Evolution du chomage par niveau d'etudes\nRegion de Bruxelles-Capitale", 
                 fontweight='bold', fontsize=13)
    ax.set_xlabel("Annee")
    ax.set_ylabel("Nombre de demandeurs d'emploi")
    ax.legend(title="Niveau d'etudes")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.savefig('02_evolution_niveau_etudes.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Graphique sauvegarde.")
else:
    print("Colonnes disponibles :", df.columns.tolist())

## 4. Comparaison communale : derniere annee disponible

In [ ]:
# Heatmap des communes par genre pour la derniere annee
derniere_annee = df['Année'].max()
df_last = df[df['Année'] == derniere_annee].copy()

fig, ax = plt.subplots(figsize=(14, 8))

# Top 10 communes par nombre total de demandeurs
df_top = df_last.nlargest(10, 'Total commune')
x = range(len(df_top))
width = 0.35

bars_h = ax.bar([i - width/2 for i in x], df_top['Total hommes'], 
                width, label='Hommes', color='steelblue', alpha=0.85)
bars_f = ax.bar([i + width/2 for i in x], df_top['Total femmes'], 
                width, label='Femmes', color='coral', alpha=0.85)

ax.set_title(f"Top 10 communes par demandeurs d'emploi en {int(derniere_annee)}\nRepartition par genre", 
             fontweight='bold', fontsize=13)
ax.set_xlabel("Commune")
ax.set_ylabel("Nombre de demandeurs d'emploi")
ax.set_xticks(list(x))
ax.set_xticklabels(df_top['Commune'], rotation=45, ha='right')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('03_comparaison_communes.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Graphique sauvegarde — Annee {int(derniere_annee)}")

## 5. Interpretation politique et conclusions

### Constats principaux

1. **Feminisation du chomage** : Les donnees montrent que les femmes representent une part significative des demandeurs d'emploi inoccupes, avec des variations selon les communes et les periodes economiques.

2. **Impact du niveau d'etudes** : Le chomage touche disproportionnellement les personnes a faible niveau d'etudes, confirmant le lien fort entre qualification et employabilite dans la region bruxelloise.

3. **Disparites communales** : Certaines communes enregistrent des taux de chomage structurellement plus eleves, souvent correlees a des quartiers historiquement defavorises.

4. **Effet COVID-19** : La periode 2020-2021 montre une rupture dans les tendances, avec un rebond notable du chomage qui affecte differemment hommes et femmes.

### Question politique repondue

> Les femmes peu qualifiees sont-elles plus vulnerables au chomage a Bruxelles ?

**Reponse** : Oui. Les donnees confirment que les femmes avec un faible niveau d'etudes represented le groupe le plus expose structurellement. Cette vulnerabilite appelle des politiques ciblees de formation et d'insertion professionnelle.

### Recommandations politiques

- Renforcer les programmes de formation qualifiante pour les femmes peu diplomees dans les communes les plus touchees.
- Developper des structures de garde d'enfants abordables pour faciliter le retour a l'emploi des femmes.
- Mettre en place des incitants specifiques pour l'embauche de personnes peu qualifiees aupres des entreprises bruxelloises.

---

**Contact pour decisions politiques ou formations IA & Data** :  
Kuate Joel Parfait — hello@dhcompany.pro | 0465 55 71 09  
[linkedin.com/in/joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
www.axiatechnologie.com